----
## <font color='CornflowerBlue'>Week 6: Calibration of LLMs</font> 
##### Marieke Smith, Alok Bharadwaj, and Arjen Jakobi
---

Last week, you learnt that language models are effective at capturing syntactic relationships between between tokens across sequences of arbitrary length. This ability makes language models especially well suited for generating sequences under a given context. In natural language, one of the most prominent use casesis the development of chatbots. 

Language models generate text through __auto-regressive next-token prediction__. At each step, the model predicts a probability distribution over the entire vocabulary and samples one token from that distribution. The probability assigned to each token depends on patterns learned during pre-training, as well as on any fine-tuning or reinforcement learning applied after the pre-training stage.

This mechanism supports a wide range of applications. For creative applications, the generated sequence may be novel, and this novelty may even be the desired outcome. However, one of the most common uses of chatbots is __information retrieval__. In such applications, the generated output should ideally align with factual information and objective descriptions of reality.

For this reason, it is useful to look not only at the token selected by the model, but also at the probabilities assigned to alternative tokens. These probabilities provide insight into the model’s uncertainty. A model that assigns high probability to one answer and very low probability to the alternatives is expressing strong confidence. By contrast, a model that distributes probability more evenly across several possible answers is indicating uncertainty. Showing these probabilities can therefore support more honest outputs. instead of presenting every answer as equally certain, the model can reveal when its prediction is confident, ambiguous, or unreliable.

In this notebook, we will study the probabilities of generated tokens using a constrained example. Multiple-choice questions (MCQs) offer a useful setting for examining whether a model is well calibrated when recovering factual information. The model can be prompted to produce a structured output, which can then be parsed and analysed.

Run the following cell to load the necessary packages. 

In [ ]:
# #@title Click here to initialize the environment (Required) { display-mode: "form" }
import os, sys, subprocess

print("Initializing environment. Please wait.")

# Clone the git repo 
subprocess.run(["git", "clone", "https://github.com/cryoTUD/ai4nanobiology.git"], stdout=subprocess.DEVNULL)
os.chdir("ai4nanobiology/week_6")
sys.path.append(os.path.abspath('.'))

# Installing requirements
subprocess.run(["pip", "install", "-q", "sentencepiece", "protobuf", "tiktoken", "transformers", "accelerate"])
print("Environment ready! You can now start the exercise.")

In [ ]:
import re
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.utils import setup_client

url = setup_client()
print("Proxy URL:", url)

### Using open weight models through Hugging Face

In this notebook, we will use the open weight _Llama-1B_ and _Llama-8B_ models. Because these models are large, running inference on local machines can be cumbersome and computationally expensive. Instead, we will use the inference infrastructure provided by **Hugging Face**.

The next code cell defines a helper function for sending prompts to the model and retrieving its output. Read through the function definition before running the cell.

In [ ]:
def query_model(prompt, model_name, max_tokens, client, system_prompt=""):
    """Send one prompt to the proxy and return the model's response.

    Parameters
    ----------
    prompt        : str   the user prompt (the question or batch of questions)
    model_name    : str   "llama-1b", or "llama-8b" 
    max_tokens    : int   how many tokens to generate
    client        : str   proxy URL from setup_client()
    system_prompt : str   optional system instruction

    Returns
    -------
    dict with keys:
        "answer"      : str   the generated text
        "logprobs"    : dict  {token: logprob}
        "token_probs" : dict  {token: probability}
        "logprob_contents" : list of dictionaries for each generated token[{}]
    or None if the request was rate-limited / rejected.
    """
    response = requests.post(
        f"{client}/generate",
        json={
            "system_prompt": system_prompt,
            "prompt": prompt,
            "model": model_name,
            "max_tokens": max_tokens,
        },
        timeout=60,
    )

    if response.status_code == 429:
        print("Rate limit hit - wait a moment and try again.")
        return None
    if response.status_code == 400:
        print(f"Bad request: {response.json().get('detail')}")
        return None

    response.raise_for_status()
    return response.json()

In [ ]:
# Models available through the proxy. Swap MODEL_NAME to compare sizes later.
MODEL_1B = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_8B = "meta-llama/Llama-3.1-8B-Instruct"

MODEL_NAME = MODEL_1B   

# Map the dataset's integer answer (0-3) to a letter (A-D)
ANSWER_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}

### 1. The MMLU Dataset
Let us first examine the structure of our multiple-choice question dataset. We will use the [MMLU dataset](https://huggingface.co/datasets/cais/mmlu), which contains questions and answer options across a wide range of academic and professional subjects.

Each example consists of a question, several answer options, and the correct answer. This structure makes MMLU useful for testing whether a language model can recover factual information in a constrained setting: instead of generating a free-form response, the model must select one answer from a fixed set of options.

In the next cell, we load a subset of the dataset and inspect how the questions and answers are represented.

In [ ]:
# Load one MMLU subject (college_biology) so we have a question to play with.
DATASET_NAME = "college_biology"
BASE_URL = "hf://datasets/cais/mmlu/"

def load_subject(dataset_name, split="test"):
    path = f"{dataset_name}/{split}-00000-of-00001.parquet"
    return pd.read_parquet(BASE_URL + path)

df_full = load_subject(DATASET_NAME)
print(f"Loaded {len(df_full)} questions from '{DATASET_NAME}'.")

Each row in MMLU represents one multiple-choice question. The most important columns are:

- `question`: the question text;
- `choices`: a list containing the four answer options;
- `answer`: an integer from 0 to 3 indicating the correct option.

We map these integer labels to the letters A, B, C, and D so that the model output can be compared directly with the dataset answer.

In [ ]:
df_full.head()

In [ ]:
df_full.iloc[3]

## 2. Prompting the Model

Next, we construct a prompt that presents one MMLU question to the model. The prompt should include the question, the four answer options, and a clear instruction about the expected output format.

Your task is to write an instruction that encourages the model to answer with only a single letter: `A`, `B`, `C`, or `D`. A precise instruction is important because it makes the model output easier to parse and compare with the correct answer.

For example, the function below should return a prompt with this structure:

```python
question text
A. first option
B. second option
C. third option
D. fourth option
```
Write your instructions in the code below: 

```python
def build_prompt(question, choices):
    option_lines = [f"{letter}. {text}"
                    for letter, text in zip("ABCD", choices)]
    instruction = "" # <-- Write your instruction so the model only answers with a single letter
    return f"{question}\n" + "\n".join(option_lines) + "\n\n" + instruction
```

In [ ]:
def build_prompt(question, choices):
    option_lines = [f"{letter}. {text}"
                    for letter, text in zip("ABCD", choices)]
    # write your own instruction below in plain English
    instruction = # TO DO 
    return f"{question}\n" + "\n".join(option_lines) + "\n\n" + instruction

In [ ]:
# Pick one question
q_idx = 3
row = df_full.iloc[q_idx]
correct_letter = ANSWER_MAP[row["answer"]]

prompt = build_prompt(row["question"], row["choices"])

print("-" * 60)
print(prompt)
print("-" * 60)
print(f"Correct answer (from our dataset): {correct_letter}")

Now we send the prompt to the model. Since the answer should be only one letter, we request only a small number of generated tokens.

After the model responds, we inspect both the raw answer and the probabilities assigned to the generated tokens. These probabilities will later be used to study confidence and calibration.

In [ ]:
# Ask the model. We only need a couple of tokens for a single letter.
result = query_model(prompt, MODEL_NAME, max_tokens=5, client=url)

print(f"Model's raw answer: {result['answer']!r}")
print(f"Correct answer    : {correct_letter}")
print()
print("Per-token probabilities the model assigned:")
for tok, p in result["token_probs"].items():
    print(f"  {tok!r:8} -> {p:.4f}")

Ideally, the model follows your instruction and returns only a single letter, possibly followed by whitespace or an end-of-text token. If the output contains extra words or an explanation, try making your instruction more explicit and run the previous cells again.

The field `result["token_probs"]` contains the probabilities associated with the generated tokens. These probabilities are computed from the model logits and provide a first indication of how confident the model was in its response.

## 3. Parsing the model response
The model output is a string, but for evaluation we need a clean answer label: `A`, `B`, `C`, or `D`. Even when the prompt asks for a single letter, the model may include extra whitespace or additional text.

To make the evaluation robust, we use a regular expression to extract the first standalone answer letter from the model response.


In [ ]:
def parse_single_letter(text):
    """Return the first standalone A/B/C/D found in text, or None."""
    match = re.search(r"\b([ABCD])\b", text.strip().upper())
    return match.group(1) if match else None


model_letter = parse_single_letter(result["answer"])
is_correct = (model_letter == correct_letter)

print(f"Parsed model answer: {model_letter}")
print(f"Correct answer     : {correct_letter}")
print(f"Correct?           : {is_correct}")

## 4. Evaluating the Model on Multiple Questions

So far, we have tested the model on a single question. To estimate performance more reliably, we need to evaluate the model on many questions and compute summary statistics such as accuracy.

Sending one request per question can be inefficient and may run into rate limits on the shared inference service. To reduce the number of requests, we group several questions into a single prompt and ask the model to return one answer per question.

For batching to work, the output format must be highly structured. The next cells create batched prompts, query the model, parse the returned answers, and compare them with the correct answer key.

In [ ]:
def batch_questions(df, batch_size):
    """Split df into batches; each batch is (prompt_string, {qnum: correct_letter})."""
    prompts, answer_keys = [], []
    for start in range(0, len(df), batch_size):
        batch = df.iloc[start:start + batch_size]
        header = (
            f"Answer the following {len(batch)} multiple choice questions.\n"
            "For each question, answer with the question number followed by a "
            "letter (A, B, C, or D) and nothing else.\n\n"
        )
        body = ""
        key = {}
        for local_i, (_, row) in enumerate(batch.iterrows(), start=1):
            option_lines = [f"{letter}. {text}"
                            for letter, text in zip("ABCD", row["choices"])]
            body += f"{local_i}: {row['question']}\n" + "\n".join(option_lines) + "\n\n"
            key[local_i] = ANSWER_MAP[row["answer"]]
        prompts.append(header + body)
        answer_keys.append(key)
    return prompts, answer_keys

In [ ]:
BATCH_SIZE = 10
prompts, answer_keys = batch_questions(df_full, batch_size=BATCH_SIZE)
print(f"{len(df_full)} questions -> {len(prompts)} batches of up to {BATCH_SIZE}.")

# Look at one batched prompt
print("\n" + "-" * 60)
print(prompts[0])

### Adding a System Prompt

A system prompt gives the model high-level instructions about how it should behave. Here, we use it to make the model act like an answer key: it should return only numbered answers and avoid explanations, markdown, or extra text.

This stricter formatting makes the batched response easier to parse automatically.

In [ ]:
SYSTEM_PROMPT = """You are an answer key. You receive multiple choice questions and output only the answers.

RULES:
- Output one line per question.
- Each line is exactly: the question number, a colon, a space, and a single capital letter (A, B, C, or D).
- Output nothing else: no explanations, no restating the question, no extra words, no blank lines, no markdown.
- Answer every question. If unsure, still pick the single most likely letter.

EXAMPLE REPLY (for 3 questions):
1: A
2: C
3: B
"""


In [ ]:
# Query the first batch. Allow enough tokens for all the numbered answers.
batch_result = query_model(
    prompts[0], MODEL_NAME, max_tokens=200, client=url, system_prompt=SYSTEM_PROMPT
)
print("Model's batched answer:\n")
print(batch_result["answer"])

### Parsing Batched Answers

The batched response should contain one line per question, such as `3: B`. We parse the full block of text into a dictionary of the form `{question_number: answer_letter}`.

This allows us to compare each model answer with the corresponding correct answer in the batch.

In [ ]:
def parse_batched_answers(text):
    """Parse 'N: X' lines into {N: 'X'} keeping only valid A-D letters."""
    answers = {}
    for line in text.strip().splitlines():
        m = re.match(r"\s*(\d+)\s*[:.]\s*([ABCD])", line.strip().upper())
        if m:
            answers[int(m.group(1))] = m.group(2)
    return answers


parsed = parse_batched_answers(batch_result["answer"])
print("Parsed:", parsed)

In [ ]:
def score_batch(parsed, answer_key, verbose=False):
    """Return accuracy of one parsed batch against its answer key."""
    correct = 0
    for qnum, gold in answer_key.items():
        got = parsed.get(qnum)
        if got == gold:
            correct += 1
        if verbose:
            mark = "OK" if got == gold else " X"
            print(f"{mark}\tQ{qnum}: correct={gold}, model={got}")
    return correct / len(answer_key)


acc = score_batch(parsed, answer_keys[0], verbose=True)
print(f"\nBatch accuracy: {acc:.1%}")

In [ ]:
def evaluate_dataset_batched(df, model_name, batch_size=10, system_prompt=SYSTEM_PROMPT,
                             max_tokens=200, pause=0.5):
    prompts, answer_keys = batch_questions(df, batch_size=batch_size)
    n_correct, n_total = 0, 0
    for prompt, key in tqdm(list(zip(prompts, answer_keys)), desc="Batches"):
        res = query_model(prompt, model_name, max_tokens=max_tokens,
                          client=url, system_prompt=system_prompt)
        time.sleep(pause)  # be gentle on the shared rate limit
        if res is None:
            continue
        parsed = parse_batched_answers(res["answer"])
        for qnum, gold in key.items():
            n_total += 1
            if parsed.get(qnum) == gold:
                n_correct += 1
    return n_correct / n_total if n_total else 0.0


overall_acc = evaluate_dataset_batched(df_full, MODEL_NAME, batch_size=BATCH_SIZE)
print(f"\nOverall accuracy on '{DATASET_NAME}' ({MODEL_NAME}): {overall_acc:.1%}")

## Calibration: Can we trust the model’s confidence?

Accuracy tells us how often the model chooses the correct answer, but it does not tell us whether the model’s confidence is reliable. For that, we study **calibration**.

A model is **well calibrated** if its confidence matches its empirical accuracy. For example, among all answers to which the model assigns about 70% confidence, approximately 70% should be correct.

We visualize calibration by plotting the model’s token probability on the x-axis and the observed accuracy on the y-axis. A perfectly calibrated model lies on the diagonal. We summarize miscalibration using the **Expected Calibration Error (ECE)**, which measures the average gap between confidence and accuracy across confidence bins, weighted by the number of examples in each bin.


In [ ]:
from src.calibration_utils import (
    collect_confidences, 
    compute_calibration,
    plot_calibration,
    calibration_for
)

### 5a. Calibration for One Subject

We first run the full calibration pipeline on one MMLU subject. While testing the notebook, keep `max_questions` relatively small so that the code runs quickly and respects the shared rate limit. For a more stable estimate, increase the number of questions or set `max_questions=None` to use the full subject.

In [ ]:
N_BINS = 10

cal_df = collect_confidences(df_full, MODEL_NAME, client=url, max_questions=50)
print(f"Collected {len(cal_df)} answered questions.")
print(f"Accuracy: {cal_df['is_correct'].mean():.1%}")

cal, ece = compute_calibration(cal_df, n_bins=N_BINS)


In [ ]:
title_text = f"Calibration - {MODEL_NAME}\n{DATASET_NAME} (N={len(cal_df)})"
plot_calibration(cal, ece, title=title_text)

### Interpreting the Calibration Plot

The dashed diagonal represents perfect calibration. Points close to this line indicate that the model’s confidence is aligned with its observed accuracy.

Points **below** the diagonal indicate **overconfidence**: the model assigns higher probabilities than its accuracy justifies. Points **above** the diagonal indicate **underconfidence**: the model is more accurate than its stated confidence suggests.

The label `N=` shows how many questions fall into each confidence bin. Bins with few questions are noisier and should be interpreted with more caution.

### 5b. Comparing calibration across model sizes

Finally, we compare the 1B and 8B Llama models on the same MMLU subjects. The larger 8B model has more parameters.

This lets us ask two related questions:

1. Does the larger model answer more questions correctly?
2. Does the larger model produce probabilities that are better calibrated?

Run the next cell to generate calibration plots and summary statistics for each model-subject combination.

In [ ]:
def calibration_for(dataset_name, model_name, client, n_bins=10, max_questions=None):
    """Full pipeline for one (subject, model): load -> query -> plot."""
    df = load_subject(dataset_name)
    res_df = collect_confidences(df, model_name, client=client, max_questions=max_questions)
    if len(res_df) == 0:
        print(f"No results for {dataset_name} / {model_name}.")
        return None
    cal, ece = compute_calibration(res_df, n_bins=n_bins)
    short = model_name.split("/")[-1]
    plot_calibration(cal, ece,
                     f"Calibration - {short}\n{dataset_name} "
                     f"(N={len(res_df)}, acc={res_df['is_correct'].mean():.0%})")
    return {"dataset": dataset_name, "model": model_name,
            "n": len(res_df), "accuracy": res_df["is_correct"].mean(), "ece": ece}

In [ ]:
# Compare model size on the two requested subjects.
# Keep max_questions modest to respect the shared rate limit; raise it for sharper plots.
SUBJECTS = ["college_biology", "college_physics"]
MODELS = [MODEL_1B, MODEL_8B]

summary = []
for subject in SUBJECTS:
    for model in MODELS:
        out = calibration_for(subject, model, client=url, n_bins=N_BINS)
        if out:
            summary.append(out)

pd.DataFrame(summary)

#### Looking ahead
In the next notebook, you will move beyond multiple-choice questions and explore how to study model confidence for open-ended responses. This is more challenging because open-ended answers are harder to parse and may have many valid forms, but the same core idea remains: useful language models should not only produce answers, but also communicate uncertainty in a meaningful way.